# Clase 200 — Kubernetes para servir modelos

Notebook **declarativo**: genera los 5 manifests YAML mínimos para desplegar el `iris-api` en K8s. Para verlos en vivo:

```bash
kind create cluster --name ml
kind load docker-image iris-api:v1 --name ml
kubectl apply -f k8s/
kubectl port-forward svc/iris-api 8000:80
```

In [ ]:
import os, shutil, tempfile
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'k8s_demo'
if WORK.exists(): shutil.rmtree(WORK)
(WORK / 'k8s').mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

## 1. `Deployment` — 3 réplicas con probes y resources

In [ ]:
deploy = '''\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api
  labels: { app: iris-api }
spec:
  replicas: 3
  strategy:
    type: RollingUpdate
    rollingUpdate: { maxSurge: 1, maxUnavailable: 0 }
  selector:
    matchLabels: { app: iris-api }
  template:
    metadata:
      labels: { app: iris-api }
    spec:
      containers:
        - name: api
          image: iris-api:v1
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          resources:
            requests: { cpu: "100m", memory: "256Mi" }
            limits:   { cpu: "500m", memory: "512Mi" }
          startupProbe:
            httpGet: { path: /health, port: 8000 }
            failureThreshold: 30      # 30 * 2s = 60s para arrancar
            periodSeconds: 2
          livenessProbe:
            httpGet: { path: /health, port: 8000 }
            periodSeconds: 10
            failureThreshold: 3
          readinessProbe:
            httpGet: { path: /health, port: 8000 }
            periodSeconds: 5
            failureThreshold: 2
          securityContext:
            runAsNonRoot: true
            runAsUser: 1000
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: false
'''
Path('k8s/deployment.yaml').write_text(deploy)
print(deploy)

## 2. `Service` — load balancer interno

In [ ]:
svc = '''\
apiVersion: v1
kind: Service
metadata:
  name: iris-api
spec:
  type: ClusterIP
  selector: { app: iris-api }
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP
'''
Path('k8s/service.yaml').write_text(svc)
print(svc)

## 3. `HorizontalPodAutoscaler` — autoescalado CPU

In [ ]:
hpa = '''\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-api
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-api
  minReplicas: 3
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: { type: Utilization, averageUtilization: 50 }
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300   # espera 5 min antes de bajar réplicas
    scaleUp:
      stabilizationWindowSeconds: 0     # sube ya
'''
Path('k8s/hpa.yaml').write_text(hpa)
print(hpa)

## 4. `Ingress` — exposición externa

In [ ]:
ing = '''\
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: iris-api
  annotations:
    nginx.ingress.kubernetes.io/limit-rps: "100"
spec:
  ingressClassName: nginx
  rules:
    - host: iris.example.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: iris-api
                port: { number: 80 }
'''
Path('k8s/ingress.yaml').write_text(ing)
print(ing)

## 5. `ConfigMap` + `Secret`

In [ ]:
cm = '''\
apiVersion: v1
kind: ConfigMap
metadata:
  name: iris-config
data:
  LOG_LEVEL: "info"
  MODEL_VERSION: "1.0.0"
---
apiVersion: v1
kind: Secret
metadata:
  name: iris-secrets
type: Opaque
stringData:
  API_KEY: "replace-me-via-external-secrets-operator"
'''
Path('k8s/config.yaml').write_text(cm)
print(cm)

## 6. Comandos kubectl de referencia

In [ ]:
commands = '''\
# Crear cluster local + cargar imagen
kind create cluster --name ml
kind load docker-image iris-api:v1 --name ml

# Apply
kubectl apply -f k8s/

# Estado
kubectl get deployments,pods,svc,hpa -l app=iris-api
kubectl describe pod -l app=iris-api
kubectl logs -l app=iris-api --tail=50

# Port-forward para probar local
kubectl port-forward svc/iris-api 8000:80
# en otra terminal: curl localhost:8000/predict -X POST -d ...

# Rolling update
kubectl set image deployment/iris-api api=iris-api:v2
kubectl rollout status deployment/iris-api
kubectl rollout history deployment/iris-api

# Rollback (instantáneo si v1 pods siguen referenciados)
kubectl rollout undo deployment/iris-api

# Force scale (puentea HPA momentáneamente)
kubectl scale deployment/iris-api --replicas=8

# Loadtest para disparar HPA
kubectl run loadtester --image=busybox -it --rm -- \\
    /bin/sh -c "while true; do wget -q -O- iris-api/health; done"
kubectl get hpa -w   # observá escalando

# Cleanup
kubectl delete -f k8s/
kind delete cluster --name ml
'''
print(commands)

## Ejercicio guiado

1. Levantá el cluster con `kind`, applies los 5 YAMLs, port-forward y curl al modelo.
2. Generá carga sintética y observá HPA escalar de 3 → 10 pods con `kubectl get hpa -w`.
3. Hacé un rolling update a una imagen rota (`iris-api:broken`). Confirmá que K8s detiene el rollout (no avanza si readiness falla). Rollback con `kubectl rollout undo`.
4. Cambiá `livenessProbe` a apuntar a `/wrong-path`. Observá `CrashLoopBackOff` en `kubectl get pods -w`. Revertí.
5. Bonus: agregá un `PodDisruptionBudget` con `minAvailable: 2` para garantizar disponibilidad durante drains de nodes.

## Conclusiones

- 5 manifests resuelven el 90% del caso ML serving: Deployment + Service + HPA + Ingress + (ConfigMap | Secret).
- Las 3 probes resuelven 3 problemas distintos — no son intercambiables.
- Sin `resources.requests`, los pods son ciudadanos de segunda; sin `limits`, son noisy neighbors.
- Rolling update + readiness probe = deploy sin downtime. Sin probe correcta = deploy con downtime invisible.